# BitCal-TTS: Bit-Calibrated Test-Time Scaling

**Full experiment on Google Colab (T4 GPU, 15 GB VRAM)**

Supports Qwen2.5 sizes:
- **3B** (~2.5 GB VRAM, ~90 min) — quick validation (100 items)
- **7B** (~4.5 GB VRAM, ~3 hr) — recommended for paper (100 items)
- **14B** (~9 GB VRAM, ~5 hr) — strongest paper results (100 items)

Optional **Gemma 2 2B** (Google, ~1.5–2 GB VRAM 4-bit) — lightweight **cross-vendor** check for the paper. Requires a free [Hugging Face token](https://huggingface.co/settings/tokens) and accepting the license on the [model card](https://huggingface.co/google/gemma-2-2b-it).

> **Safe to interrupt at any time** — results are written to disk after every single row.

**Steps:** Check GPU → Install → Clone → Smoke test → Run experiment → Analyze → Download

In [ ]:
# Cell 1 — Verify GPU and pick model
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU : {gpu_name}')
    print(f'VRAM: {vram_gb:.1f} GB')
    if vram_gb >= 12:
        print('Recommendation: 7B (fast) or 14B (best accuracy)')
    else:
        print('Recommendation: 3B (low VRAM)')
else:
    print('WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 2 — Install dependencies
!pip install -q -U bitsandbytes>=0.46.1
!pip install -q -U transformers>=4.38.0 accelerate>=0.27.0 datasets>=2.18.0 matplotlib>=3.7.0 pyyaml scipy
print('Dependencies installed.')

In [ ]:
# Cell 3 — Clone / update repo
import os, subprocess

if not os.path.exists('/content/bitcal-tts'):
    !git clone https://github.com/Saibabu7770/bitcal-tts.git /content/bitcal-tts
else:
    !git -C /content/bitcal-tts pull origin main

os.chdir('/content/bitcal-tts')
!pip install -e . -q
print(f'Repo ready. CWD: {os.getcwd()}')

In [ ]:
# Cell 4 — Verify setup
import os
os.chdir('/content/bitcal-tts')
!python -m bitcal_tts doctor

In [ ]:
# Cell 5 — Smoke test (1 item, fixed only, confirms model loads)
import os
os.chdir('/content/bitcal-tts')

# Choose model for smoke test (always use 3B for speed)
!python scripts/run_experiment.py \
    --model Qwen/Qwen2.5-3B-Instruct \
    --n-samples 1 \
    --budget 64 \
    --methods fixed \
    --source hf \
    --step-size 8 \
    --output-dir results/raw
print('Smoke test done.')

## Cell 6A / 6B / 6C / 6G — Choose your model and run

Run **only one** of the experiment cells below depending on how much time you have.
Qwen cells run **100 items** when fully finished (900 rows = 100 × 3 methods × 3 budgets).
**Safe to interrupt at any time** — every row is saved to disk immediately.

| Cell | Model | VRAM | Time (100 items) | Best for |
|---|---|---|---|---|
| 6A | Qwen 3B | 2.5 GB | ~90 min | Quick re-run / validation |
| **6B** | **Qwen 7B** | **4.5 GB** | **~3 hr** | **Recommended for paper** |
| 6C | Qwen 14B | 9 GB | ~5 hr | Strongest paper results |
| **6G** | **Gemma 2 2B** | **~2 GB** | **~60–90 min (100 items)** | Same params as Qwen (`experiment_gsm8k_gemma2_2b.yaml`) |

In [ ]:
# Cell 6A — Qwen2.5-3B-Instruct 4-bit (~90 min, 100 items)
# Safe to interrupt — each row is saved to disk immediately.
import os
os.chdir('/content/bitcal-tts')

!python scripts/run_experiment.py \
    --model Qwen/Qwen2.5-3B-Instruct \
    --n-samples 100 \
    --budget 256 512 1024 \
    --methods fixed adaptive bitcal_tts \
    --source hf \
    --step-size 16 \
    --min-tokens-before-halt 128 \
    --bit-width 4 \
    --output-dir results/raw
print('3B experiment complete.')

In [ ]:
# Cell 6B — Qwen2.5-7B-Instruct 4-bit (~3 hr, 100 items) RECOMMENDED FOR PAPER
# Safe to interrupt — each row is saved to disk immediately.
# You will see: [1/900] ... [2/900] ... — stop any time and download results.
import os
os.chdir('/content/bitcal-tts')

!python scripts/run_experiment.py \
    --model Qwen/Qwen2.5-7B-Instruct \
    --n-samples 100 \
    --budget 256 512 1024 \
    --methods fixed adaptive bitcal_tts \
    --source hf \
    --step-size 16 \
    --min-tokens-before-halt 128 \
    --bit-width 4 \
    --output-dir results/raw
print('7B experiment complete.')

In [ ]:
# Cell 6C — Qwen2.5-14B-Instruct 4-bit (~5 hr, 100 items) BEST ACCURACY
# Safe to interrupt — each row is saved to disk immediately.
import os
os.chdir('/content/bitcal-tts')

!python scripts/run_experiment.py \
    --model Qwen/Qwen2.5-14B-Instruct \
    --n-samples 100 \
    --budget 512 1024 \
    --methods fixed adaptive bitcal_tts \
    --source hf \
    --step-size 16 \
    --min-tokens-before-halt 128 \
    --bit-width 4 \
    --output-dir results/raw
print('14B experiment complete.')

### Cell 6G — Gemma 2 2B (cross-vendor experiment)

**Before running:** (1) Create a Hugging Face account and [access token](https://huggingface.co/settings/tokens). (2) Open [google/gemma-2-2b-it](https://huggingface.co/google/gemma-2-2b-it), click **Agree and access repository**.

**Colab:** paste your token in the next cell (`HF_TOKEN = "hf_..."`) or use **Secrets** (`HF_TOKEN`) and the code below will pick it up.

Parameters match Qwen runs: **100 items**, budgets **256 / 512 / 1024**, `step_size=16`, `min_tokens_before_halt=128`, **4-bit**, same policy thresholds. Config file: `configs/experiment_gsm8k_gemma2_2b.yaml`. Chat template is enabled in that config for proper Gemma instruct formatting.

In [ ]:
# Cell 6G — google/gemma-2-2b-it (same experiment params as Qwen; ~60–90 min, 900 rows)
import os
os.chdir('/content/bitcal-tts')

# Token: Colab Secrets (name HF_TOKEN) OR paste here:
try:
    from google.colab import userdata
    _tok = userdata.get('HF_TOKEN')
except Exception:
    _tok = None
HF_TOKEN = _tok or ''  # or paste: HF_TOKEN = 'hf_...'
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
else:
    print('ERROR: Set HF_TOKEN (Colab Secret or variable above) for gated Gemma.')
    raise SystemExit(1)

!python scripts/run_experiment.py --config configs/experiment_gsm8k_gemma2_2b.yaml
print('Gemma 2B experiment complete.')

In [ ]:
# Cell 7 — Analyze results and generate plots
import os
os.chdir('/content/bitcal-tts')
!python scripts/analyze_results.py \
    --results-dir results/raw \
    --out-dir results/processed

In [ ]:
# Cell 8 — Display plots inline
from IPython.display import Image, display
import os

os.chdir('/content/bitcal-tts')
for fname in ['pareto_quality_tokens.png', 'accuracy_by_budget.png']:
    path = f'results/processed/{fname}'
    if os.path.exists(path):
        print(f'\n--- {fname} ---')
        display(Image(path))

In [ ]:
# Cell 9 — Print full summary table
import pandas as pd, os
os.chdir('/content/bitcal-tts')
df = pd.read_csv('results/processed/summary.csv')
# Show only rows from the latest large experiment (n>=50)
df_main = df[df['n'].astype(int) >= 50]
print('=== Main results (n >= 50 items) ===')
print(df_main.to_string(index=False))
print('\n=== Full table ===')
print(df.to_string(index=False))

In [ ]:
# Cell 10 — Download all results as a zip
import shutil, os
from google.colab import files

os.chdir('/content/bitcal-tts')
shutil.make_archive('bitcal_tts_results', 'zip', '.', 'results')
files.download('bitcal_tts_results.zip')
print('Downloaded: bitcal_tts_results.zip')

## Optional: Push results back to GitHub

Create a PAT at https://github.com/settings/tokens (Contents: Read and write)

In [ ]:
# Cell 11 (optional) — Push results to GitHub
import os
os.chdir('/content/bitcal-tts')

GITHUB_TOKEN = ''  # paste your PAT here
GITHUB_EMAIL = 'Saibabu7770@users.noreply.github.com'

if GITHUB_TOKEN:
    !git config user.name 'Sai'
    !git config user.email '{GITHUB_EMAIL}'
    !git remote set-url origin https://{GITHUB_TOKEN}@github.com/Saibabu7770/bitcal-tts.git
    !git add results/
    !git commit -m 'Results: Colab T4 GPU experiment'
    !git push origin main
    print('Results pushed to GitHub!')
else:
    print('Skipped: set GITHUB_TOKEN above to push results.')